# Example 7.11.3 / 1: Time-Indexed Formulation

Companion notebook for **7.11.3 Scheduling Formulations** in the AMPL lab Documentation, section "Time-Indexed Formulation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Time-Indexed Formulation

Let `Start[j,t]` equal 1 when job `j` begins in period `t`. The sparse set `STARTS` keeps only
start periods that respect the release date and finish by the horizon. A unit-capacity row for
period `t` sums every start whose interval contains `t`; the half-open convention
`[start, start + duration)` lets one job begin when another finishes:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set JOBS;                           # jobs to schedule
param duration {JOBS} integer > 0;  # processing time, periods
param release {JOBS} integer >= 0;  # earliest start period
param weight {JOBS} >= 0;           # importance of finishing early
param horizon integer > 0;          # periods in the time grid
set TIMES := 0..horizon - 1;        # periods of the time grid
set STARTS := {j in JOBS, t in TIMES: t >= release[j] and t + duration[j] <= horizon};
var Start {STARTS} binary;          # 1 if the job starts in the period

minimize WeightedCompletion:
    sum {(j,t) in STARTS} weight[j] * (t + duration[j]) * Start[j,t];
# each job starts exactly once
subject to StartOnce {j in JOBS}:
    sum {(j,t) in STARTS} Start[j,t] = 1;
# unit capacity: at most one job active in period t
subject to OneAtATime {t in TIMES}:
    sum {(j,s) in STARTS: s <= t < s + duration[j]} Start[j,s] <= 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["JOBS"] = ["A", "B", "C"]
ampl.param["duration"] = {"A": 3, "B": 2, "C": 4}
ampl.param["release"] = {"A": 0, "B": 1, "C": 0}
ampl.param["weight"] = {"A": 1, "B": 3, "C": 2}
ampl.param["horizon"] = 12

In [5]:
# run.py
ampl.solve(solver="highs")
starts = {j: t for (j, t), v in ampl.var["Start"].to_dict().items() if v > 0.5}
print("starts:", starts)

HiGHS 1.14.0: optimal solution; objective 33
10 simplex iterations
1 branching nodes
starts: {'A': 7, 'B': 1, 'C': 3}


### Pure AMPL: data.dat and model.run

The machine waits for `B`, the job with the largest weight per period, until its release at 1;
then `C` runs from 3 to 7 and `A` from 7 to 10, for a weighted completion time of
`3*3 + 2*7 + 1*10 = 33`. Starting `C` at once, to avoid the idle period, would give 35.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: JOBS: duration release weight :=
  A  3  0  1
  B  2  1  3
  C  4  0  2 ;
param horizon := 12;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display {(j,t) in STARTS: Start[j,t] > 0.5};

HiGHS 1.14.0: optimal solution; objective 33
10 simplex iterations
1 branching nodes
set {(j,t) in STARTS: Start[j,t] > 0.5}  := (A,7) (B,1) (C,3);

